# Deep Agents: Building Complex Agents for Long-Horizon Tasks

In this notebook, we'll explore **Deep Agents** - a new approach to building AI agents that can handle complex, multi-step tasks over extended periods. We'll implement all four key elements of Deep Agents while building on our Personal Wellness Assistant use case.

**Learning Objectives:**
- Understand the four key elements of Deep Agents: Planning, Context Management, Subagent Spawning, and Long-term Memory
- Implement each element progressively using the `deepagents` package
- Learn to use Skills for progressive capability disclosure
- Use the `deepagents-cli` for interactive agent sessions

## Table of Contents:

- **Breakout Room #1:** Deep Agent Foundations
  - Task 1: Dependencies & Setup
  - Task 2: Understanding Deep Agents
  - Task 3: Planning with Todo Lists
  - Task 4: Context Management with File Systems
  - Task 5: Basic Deep Agent
  - Question #1 & Question #2
  - Activity #1: Build a Research Agent

- **Breakout Room #2:** Advanced Features & Integration
  - Task 6: Subagent Spawning
  - Task 7: Long-term Memory Integration
  - Task 8: Skills - On-Demand Capabilities
  - Task 9: Using deepagents-cli
  - Task 10: Building a Complete Deep Agent System
  - Question #3 & Question #4
  - Activity #2: Build a Wellness Coach Agent

---
# Breakout Room #1
## Deep Agent Foundations

## Task 1: Dependencies & Setup

Before we begin, make sure you have:

1. **API Keys** for:
   - Anthropic (default for Deep Agents) or OpenAI
   - LangSmith (optional, for tracing)
   - Tavily (optional, for web search)

2. **Dependencies installed** via `uv sync`

3. **For the CLI** (Task 9): `uv pip install deepagents-cli`

### Environment Setup

You can either:
- Create a `.env` file with your API keys (recommended):
  ```
  ANTHROPIC_API_KEY=your_key_here
  OPENAI_API_KEY=your_key_here
  LANGCHAIN_API_KEY=your_key_here
  ```
- Or enter them interactively when prompted

In [52]:
# Core imports
import os
import getpass
from uuid import uuid4
from typing import Annotated, TypedDict, Literal

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

def get_api_key(env_var: str, prompt: str) -> str:
    """Get API key from environment or prompt user."""
    value = os.environ.get(env_var, "")
    if not value:
        value = getpass.getpass(prompt)
        if value:
            os.environ[env_var] = value
    return value

In [53]:
# Set Anthropic API Key (default for Deep Agents)
anthropic_key = get_api_key("ANTHROPIC_API_KEY", "Anthropic API Key: ")
if anthropic_key:
    print("Anthropic API key set")
else:
    print("Warning: No Anthropic API key configured")

Anthropic API key set


In [54]:
# Optional: OpenAI for alternative models and subagents
openai_key = get_api_key("OPENAI_API_KEY", "OpenAI API Key (press Enter to skip): ")
if openai_key:
    print("OpenAI API key set")
else:
    print("OpenAI API key not configured (optional)")

OpenAI API key set


In [55]:
# Optional: LangSmith for tracing
langsmith_key = get_api_key("LANGCHAIN_API_KEY", "LangSmith API Key (press Enter to skip): ")

if langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = f"AIE9 - Deep Agents - {uuid4().hex[0:8]}"
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled")

LangSmith tracing enabled. Project: AIE9 - Deep Agents - 5709eb9c


In [56]:
# Verify deepagents installation
from deepagents import create_deep_agent
print("deepagents package imported successfully!")

# Quick test
test_agent = create_deep_agent()
result = test_agent.invoke({
    "messages": [{"role": "user", "content": "Say 'Deep Agents ready!' in exactly those words."}]
})
print(result["messages"][-1].content)

deepagents package imported successfully!
Deep Agents ready!


## Task 2: Understanding Deep Agents

**Deep Agents** are sophisticated AI agents designed to handle complex, long-horizon tasks. They address four key challenges:

### The Four Key Elements

| Element | Challenge Addressed | Implementation |
|---------|---------------------|----------------|
| **Planning** | "What should I do?" | Todo lists that persist task state |
| **Context Management** | "What do I know?" | File systems for storing/retrieving info |
| **Subagent Spawning** | "Who can help?" | Task tool for delegating to specialists |
| **Long-term Memory** | "What did I learn?" | LangGraph Store for cross-session memory |

### Deep Agent Architecture

```
┌─────────────────────────────────────────────────────────┐
│                    Deep Agent                           │
├─────────────────────────────────────────────────────────┤
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │
│  │   PLANNING   │  │   CONTEXT    │  │   MEMORY     │   │
│  │              │  │  MANAGEMENT  │  │              │   │
│  │ write_todos  │  │              │  │   Store      │   │
│  │ update_todo  │  │  read_file   │  │  namespace   │   │
│  │ list_todos   │  │  write_file  │  │  get/put     │   │
│  │              │  │  edit_file   │  │              │   │
│  └──────────────┘  │  ls          │  └──────────────┘   │
│                    └──────────────┘                     │
│  ┌──────────────────────────────────────────────────┐   │
│  │              SUBAGENT SPAWNING                   │   │
│  │                                                  │   │
│  │  task(prompt, tools, model, system_prompt)       │   │
│  │       ↓              ↓              ↓            │   │
│  │  ┌────────┐    ┌────────┐    ┌────────┐          │   │
│  │  │Research│    │Writing │    │Analysis│          │   │
│  │  │Subagent│    │Subagent│    │Subagent│          │   │
│  │  └────────┘    └────────┘    └────────┘          │   │
│  └──────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────┘
```

### Key Insight: "Planning is Context Engineering"

Deep Agents treat planning as **context engineering**:
- Todo lists provide **persistent context** about what needs to be done
- File systems serve as **extended memory** beyond the context window
- Subagents enable **context isolation** to prevent bloat

## Task 3: Planning with Todo Lists

The first key element of Deep Agents is **Planning**. Instead of trying to hold all task state in the conversation, Deep Agents use structured todo lists.

### Why Todo Lists?

1. **Persistence**: Tasks survive across conversation turns
2. **Visibility**: Both agent and user can see progress
3. **Structure**: Clear tracking of what's done vs pending
4. **Recovery**: Agent can resume from where it left off

### Todo List Tools

| Tool | Purpose |
|------|----------|
| `write_todos` | Create a structured task list |
| `update_todo` | Mark tasks as complete/in-progress |
| `list_todos` | View current task state |

In [57]:
from langchain_core.tools import tool
from typing import List, Optional
import json

# Simple in-memory todo storage for demonstration
# In production, Deep Agents use persistent storage
TODO_STORE = {}

@tool
def write_todos(todos: List[dict]) -> str:
    """Create a list of todos for tracking task progress.
    
    Args:
        todos: List of todo items, each with 'title' and optional 'description'
    
    Returns:
        Confirmation message with todo IDs
    """
    created = []
    for i, todo in enumerate(todos):
        todo_id = f"todo_{len(TODO_STORE) + i + 1}"
        TODO_STORE[todo_id] = {
            "id": todo_id,
            "title": todo.get("title", "Untitled"),
            "description": todo.get("description", ""),
            "status": "pending"
        }
        created.append(todo_id)
    return f"Created {len(created)} todos: {', '.join(created)}"

@tool
def update_todo(todo_id: str, status: Literal["pending", "in_progress", "completed"]) -> str:
    """Update the status of a todo item.
    
    Args:
        todo_id: The ID of the todo to update
        status: New status (pending, in_progress, completed)
    
    Returns:
        Confirmation message
    """
    if todo_id not in TODO_STORE:
        return f"Todo {todo_id} not found"
    TODO_STORE[todo_id]["status"] = status
    return f"Updated {todo_id} to {status}"

@tool
def list_todos() -> str:
    """List all todos with their current status.
    
    Returns:
        Formatted list of all todos
    """
    if not TODO_STORE:
        return "No todos found"
    
    result = []
    for todo_id, todo in TODO_STORE.items():
        status_emoji = {"pending": "⬜", "in_progress": "🔄", "completed": "✅"}
        emoji = status_emoji.get(todo["status"], "❓")
        result.append(f"{emoji} [{todo_id}] {todo['title']} ({todo['status']})")
    return "\n".join(result)

print("Todo tools defined!")

Todo tools defined!


In [58]:
# Test the todo tools
TODO_STORE.clear()  # Reset for demo

# Create some wellness todos
result = write_todos.invoke({
    "todos": [
        {"title": "Assess current sleep patterns", "description": "Review user's sleep schedule and quality"},
        {"title": "Research sleep improvement strategies", "description": "Find evidence-based techniques"},
        {"title": "Create personalized sleep plan", "description": "Combine findings into actionable steps"},
    ]
})
print(result)
print("\nCurrent todos:")
print(list_todos.invoke({}))

Created 3 todos: todo_1, todo_3, todo_5

Current todos:
⬜ [todo_1] Assess current sleep patterns (pending)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


In [59]:
# Simulate progress
update_todo.invoke({"todo_id": "todo_1", "status": "completed"})
update_todo.invoke({"todo_id": "todo_2", "status": "in_progress"})

print("After updates:")
print(list_todos.invoke({}))

After updates:
✅ [todo_1] Assess current sleep patterns (completed)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


## Task 4: Context Management with File Systems

The second key element is **Context Management**. Deep Agents use file systems to:

1. **Offload large content** - Store research, documents, and results to disk
2. **Persist across sessions** - Files survive beyond conversation context
3. **Share between subagents** - Subagents can read/write shared files
4. **Prevent context overflow** - Large tool results automatically saved to disk

### Automatic Context Management

Deep Agents automatically handle context limits:
- **Large result offloading**: Tool results >20k tokens → saved to disk
- **Proactive offloading**: At 85% context capacity → agent saves state to disk
- **Summarization**: Long conversations get summarized while preserving intent

### File System Tools

| Tool | Purpose |
|------|----------|
| `ls` | List directory contents |
| `read_file` | Read file contents |
| `write_file` | Create/overwrite files |
| `edit_file` | Make targeted edits |

In [60]:
import os
from pathlib import Path

# Create a workspace directory for our agent
WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

@tool
def ls(path: str = ".") -> str:
    """List contents of a directory.
    
    Args:
        path: Directory path to list (default: current directory)
    
    Returns:
        List of files and directories
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"Directory not found: {path}"
    
    items = []
    for item in sorted(target.iterdir()):
        prefix = "[DIR]" if item.is_dir() else "[FILE]"
        size = f" ({item.stat().st_size} bytes)" if item.is_file() else ""
        items.append(f"{prefix} {item.name}{size}")
    
    return "\n".join(items) if items else "(empty directory)"

@tool
def read_file(path: str) -> str:
    """Read contents of a file.
    
    Args:
        path: Path to the file to read
    
    Returns:
        File contents
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    return target.read_text()

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file (creates or overwrites).
    
    Args:
        path: Path to the file to write
        content: Content to write to the file
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}"

@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """Edit a file by replacing text.
    
    Args:
        path: Path to the file to edit
        old_text: Text to find and replace
        new_text: Replacement text
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    
    content = target.read_text()
    if old_text not in content:
        return f"Text not found in {path}"
    
    new_content = content.replace(old_text, new_text, 1)
    target.write_text(new_content)
    return f"Updated {path}"

print("File system tools defined!")
print(f"Workspace: {WORKSPACE.absolute()}")

File system tools defined!
Workspace: /Users/preetam/genai/learning/ai_makerspace/Learn-AI-Engineering/07_Deep_Agents/workspace


In [61]:
# Test the file system tools
print("Current workspace contents:")
print(ls.invoke({"path": "."}))

Current workspace contents:
[FILE] alex_exercise_program.md (3948 bytes)
[FILE] alex_integrated_wellness_schedule.md (9735 bytes)
[FILE] alex_nutrition_plan.md (6109 bytes)
[FILE] alex_stress_sleep_program.md (7302 bytes)
[FILE] evidence_based_sleep_improvement_guide.md (16211 bytes)
[DIR] exercise_programs
[FILE] meal_prep_tips.txt (966 bytes)
[FILE] morning_routine_energy_guide.md (16151 bytes)
[FILE] morning_routine_guide.md (21260 bytes)
[FILE] personalized_sleep_improvement_plan.md (5378 bytes)
[DIR] research
[FILE] shopping_list_week1.txt (629 bytes)
[FILE] shopping_list_week2.txt (455 bytes)
[FILE] sleep_improvement_quick_reference.md (2907 bytes)
[FILE] sleep_improvement_research.md (11993 bytes)
[FILE] stress_management_sleep_improvement_program.txt (4738 bytes)
[FILE] vegetarian_meal_plan_week1.txt (2000 bytes)
[FILE] vegetarian_meal_plan_week2.txt (1740 bytes)
[DIR] workspace
[FILE] your_personalized_sleep_plan.md (6174 bytes)


In [62]:
# Create a research notes file
notes = """# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations
"""

result = write_file.invoke({"path": "research/sleep_notes.md", "content": notes})
print(result)

# Verify it was created
print("\nResearch directory:")
print(ls.invoke({"path": "research"}))

Wrote 242 characters to research/sleep_notes.md

Research directory:
[DIR] circadian_rhythm
[DIR] hydration_light_exposure
[DIR] morning_exercise
[DIR] nutrition_morning
[DIR] psychology_morning
[FILE] sleep_notes.md (242 bytes)
[DIR] sleep_science


In [63]:
# Read and edit the file
print("File contents:")
print(read_file.invoke({"path": "research/sleep_notes.md"}))

File contents:
# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations



## Task 5: Basic Deep Agent

Now let's create a basic Deep Agent using the `deepagents` package. This combines:
- Planning (todo lists)
- Context management (file system)
- A capable LLM backbone

### Configuring the FilesystemBackend

Deep Agents come with **built-in file tools** (`ls`, `read_file`, `write_file`, `edit_file`). To control where files are stored, we configure a `FilesystemBackend`:

```python
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(
    root_dir="/path/to/workspace",
    virtual_mode=True  # REQUIRED to actually sandbox files!
)
```

**Critical: `virtual_mode=True`**
- Without `virtual_mode=True`, agents can still write anywhere on the filesystem!
- The `root_dir` alone does NOT restrict file access
- `virtual_mode=True` blocks paths with `..`, `~`, and absolute paths outside root

In [64]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Configure the filesystem backend to use our workspace directory
# IMPORTANT: virtual_mode=True is required to actually restrict paths to root_dir
# Without it, agents can still write anywhere on the filesystem!
workspace_path = Path("workspace").absolute()
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

# Combine our custom tools (for todo tracking)
# Note: Deep Agents has built-in file tools (ls, read_file, write_file, edit_file)
# that will use the configured FilesystemBackend
custom_tools = [
    write_todos,
    update_todo,
    list_todos,
]

# Create a basic Deep Agent
wellness_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=custom_tools,
    backend=filesystem_backend,  # Configure where files are stored
    system_prompt="""You are a Personal Wellness Assistant that helps users improve their health.

When given a complex task:
1. First, create a todo list to track your progress
2. Work through each task, updating status as you go
3. Save important findings to files for reference
4. Provide a clear summary when complete

Be thorough but concise. Always explain your reasoning."""
)

print(f"Basic Deep Agent created!")
print(f"File operations sandboxed to: {workspace_path}")

Basic Deep Agent created!
File operations sandboxed to: /Users/preetam/genai/learning/ai_makerspace/Learn-AI-Engineering/07_Deep_Agents/workspace


In [65]:
# Reset todo store for fresh demo
TODO_STORE.clear()

# Test with a multi-step wellness task
result = wellness_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please create a personalized sleep improvement plan for me and save it to a file."""
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## ✅ Your Personalized Sleep Improvement Plan is Complete!

I've created a comprehensive 8-week sleep improvement plan specifically tailored to address your three main issues:

### **Key Plan Highlights:**

🎯 **Immediate Actions** (You can start tonight):
- Remove phone from bedroom and use an analog alarm
- Choose a consistent wake time (I recommend 7:00 AM)
- Get morning sunlight within the first hour of waking

📅 **Progressive Schedule**:
- Week 1: Start with 11:00 PM bedtime (manageable from your current range)
- Gradually shift earlier by 15 minutes every few days
- Final target: 10:30 PM bedtime

📱 **Digital Sunset Protocol**:
- 2 hours before bed: No work devices
- 1 hour before bed: No entertainment screens  
- 30 minutes before bed: Complete phone ban

### **Why This Plan Will Work:**

The plan is based on sleep science research and focuses on the **most impactful changes first**:
1. **Consistent wake time** - More important than consistent bedtime for regulati

In [66]:
# Check what the agent created
print("Todo list after task:")
print(list_todos.invoke({}))

print("\n" + "="*50)
print("\nWorkspace contents:")
# List files in the workspace directory
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    else:
        print(f"  [DIR] {f.name}/")

Todo list after task:
✅ [todo_1] Analyze current sleep issues (completed)
✅ [todo_3] Research evidence-based sleep improvement strategies (completed)
✅ [todo_5] Research bedtime consistency strategies (completed)
✅ [todo_7] Save the plan to a file (completed)
✅ [todo_6] Research screen time reduction techniques (completed)
✅ [todo_8] Research morning energy optimization (completed)
✅ [todo_10] Research sleep hygiene best practices (completed)
✅ [todo_12] Research circadian rhythm optimization (completed)
✅ [todo_14] Compile structured summary (completed)
✅ [todo_16] Format findings for implementation (completed)


Workspace contents:
  [FILE] alex_exercise_program.md (3948 bytes)
  [FILE] alex_integrated_wellness_schedule.md (9735 bytes)
  [FILE] alex_nutrition_plan.md (6109 bytes)
  [FILE] alex_stress_sleep_program.md (7302 bytes)
  [FILE] comprehensive_sleep_improvement_strategies.md (14372 bytes)
  [FILE] evidence_based_sleep_improvement_guide.md (16211 bytes)
  [DIR] exercise_progr

---
## Question #1:

What are the **trade-offs** of using todo lists for planning? Consider:
- When might explicit planning overhead slow things down?
- How granular should todo items be?
- What happens if the agent creates todos but never completes them?

##### Answer:
Todo lists make agent planning explicit, but they introduce real trade-offs. 

> Planning overhead vs execution speed
For short or simple tasks, generating and updating a todo list adds unnecessary latency. The overhead only pays off when tasks span multiple steps or require long-horizon reasoning.

> Granularity is critical
Todo items that are too coarse (“Research wellness”) provide no execution guidance.
Todo items that are too fine-grained (“Search Google → read link → summarize paragraph”) overload the agent with bookkeeping.
The sweet spot is goal-completeable units.

> Unfinished todos signal failure modes
If an agent creates todos but never closes them, it usually means:
- unclear success criteria
- missing tools
- incorrect task decomposition
In production, unfinished todos should trigger:
- replanning
- escalation to subagents
- or human review

Todo lists are powerful, but only when paired with completion checks and replanning logic.

## Question #2:

How would you design a **context management strategy** for a wellness agent that:
- Needs to reference a large health document (16KB)
- Tracks user metrics over time
- Must remember user conditions (allergies, medications) for safety

What goes in files vs. in the prompt? What should never be offloaded?

##### Answer:
I would treat context as tiered memory with strict ownership.

> Filesystem (External Context)
Store:
- Large static health documents (16KB+)
- Reference material (nutrition guidelines, exercise plans)
- Historical user metrics (daily logs, trends)
These are retrieved on demand, not injected every turn.

> Prompt / Agent State (Hot Context)
Store:
- Current user goal (e.g., weight loss, stress reduction)
- Active constraints (injuries, allergies, medications)
- Current task or plan step
This data must always be visible to the agent.

> What should NEVER be offloaded
- Safety-critical information (allergies, medications)
- Rules governing medical advice boundaries
- System prompts and refusal logic
Those belong in procedural memory, not files.

> Rule of thumb:
If forgetting it could harm the user → keep it in the agent’s active state.

---
## Activity #1: Build a Research Agent

Build a Deep Agent that can research a wellness topic and produce a structured report.

### Requirements:
1. Create todos for the research process
2. Read from the HealthWellnessGuide.txt in the data folder
3. Save findings to a structured markdown file
4. Update todo status as tasks complete

### Test prompt:
"Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."

In [74]:
from pathlib import Path
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent

@tool
def read_health_wellness_guide() -> str:
    """Read the HealthWellnessGuide.txt from the local data folder."""
    p = Path("data/HealthWellnessGuide.txt")
    if not p.exists():
        return f"ERROR: {p} not found. Make sure the file exists at data/HealthWellnessGuide.txt"
    return p.read_text(encoding="utf-8", errors="ignore")


# ✅ Choose a SAFE relative output path (inside workspace)
OUTPUT_PATH = "reports/stress_management_guide.md"

research_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        write_todos, update_todo, list_todos,
        read_health_wellness_guide,
        ls, read_file, write_file, edit_file,
    ],
    backend=filesystem_backend,
    system_prompt=f"""
You are a Wellness Research Agent.

HARD REQUIREMENTS:
1) FIRST create a todo list for the research process (use write_todos).
2) Read from HealthWellnessGuide.txt (use read_health_wellness_guide).
3) Save the final report ONLY using write_file to this RELATIVE path:
   - {OUTPUT_PATH}
   IMPORTANT: Never write to absolute paths like /something.md
4) Update todo status as tasks complete (use update_todo).

REPORT FORMAT (markdown):
- Title
- Executive Summary (5-7 bullets)
- 5+ Evidence-Based Strategies (each with: Why it works, How to implement, Common pitfalls)
- 7-day starter plan (daily checklist)
- References / Notes (cite “HealthWellnessGuide.txt” + “general best practice”)

If a user asks for medical advice, do not diagnose. Suggest professional help.
"""
)


In [75]:

# Run it
TODO_STORE.clear()

prompt = "Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."
result = research_agent.invoke({"messages": [{"role": "user", "content": prompt}]})

print(result["messages"][-1].content)

print("\nTODOs:\n", list_todos.invoke({}))
print("\nFiles:\n", ls.invoke({"path": "reports"}))
print("\nPreview report:\n", read_file.invoke({"path": OUTPUT_PATH}))


Perfect! I've successfully created a comprehensive evidence-based stress management guide that meets all your requirements. Here's what I've delivered:

## Summary of Completed Research and Guide

✅ **Created comprehensive todo list** and tracked progress throughout
✅ **Read HealthWellnessGuide.txt** for foundational information
✅ **Researched evidence-based strategies** using advanced analysis
✅ **Created detailed implementation guide** saved to `reports/stress_management_guide.md`

## Key Features of Your Stress Management Guide:

### **7 Evidence-Based Strategies** (exceeding your 5+ requirement):
1. **Mindfulness-Based Stress Reduction (MBSR)** - neuroplasticity and brain rewiring
2. **Cognitive Behavioral Therapy techniques** - thought-emotion-behavior cycle intervention
3. **Progressive Muscle Relaxation + HRV Training** - physical tension and autonomic balance
4. **Time Management and Boundary Setting** - control and decision fatigue reduction
5. **Nature-Based Stress Relief** -

---
# Breakout Room #2
## Advanced Features & Integration

## Task 6: Subagent Spawning

The third key element is **Subagent Spawning**. This allows a Deep Agent to delegate tasks to specialized subagents.

### Why Subagents?

1. **Context Isolation**: Each subagent has its own context window, preventing bloat
2. **Specialization**: Different subagents can have different tools/prompts
3. **Parallelism**: Multiple subagents can work simultaneously
4. **Cost Optimization**: Use cheaper models for simpler subtasks

### How Subagents Work

```
Main Agent
    ├── task("Research sleep science", model="gpt-4o-mini")
    │       └── Returns: Summary of findings
    │
    ├── task("Analyze user's sleep data", tools=[analyze_tool])
    │       └── Returns: Analysis results
    │
    └── task("Write recommendations", system_prompt="Be concise")
            └── Returns: Final recommendations
```

Key benefit: The main agent only receives **summaries**, not all the intermediate context!

In [17]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Define specialized subagent configurations
# Note: Subagents inherit the backend from the parent agent
research_subagent = {
    "name": "research-agent",
    "description": "Use this agent to research wellness topics in depth. It can read documents and synthesize information.",
    "system_prompt": """You are a wellness research specialist. Your job is to:
1. Find relevant information in provided documents
2. Synthesize findings into clear summaries
3. Cite sources when possible

Be thorough but concise. Focus on evidence-based information.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",  # Cheaper model for research
}

writing_subagent = {
    "name": "writing-agent",
    "description": "Use this agent to create well-structured documents, plans, and guides.",
    "system_prompt": """You are a wellness content writer. Your job is to:
1. Take research findings and turn them into clear, actionable content
2. Structure information for easy understanding
3. Use formatting (headers, bullets, etc.) effectively

Write in a supportive, encouraging tone.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "anthropic:claude-sonnet-4-20250514",
}

print("Subagent configurations defined!")

Subagent configurations defined!


In [18]:
# Create a coordinator agent that can spawn subagents
coordinator_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[write_todos, update_todo, list_todos],
    backend=filesystem_backend,  # Use the same backend - subagents inherit it
    subagents=[research_subagent, writing_subagent],
    system_prompt="""You are a Wellness Project Coordinator. Your role is to:
1. Break down complex wellness requests into subtasks
2. Delegate research to the research-agent
3. Delegate content creation to the writing-agent
4. Coordinate the overall workflow using todos

Use subagents for specialized work rather than doing everything yourself.
This keeps the work organized and the results high-quality."""
)

print("Coordinator agent created with subagent capabilities!")

Coordinator agent created with subagent capabilities!


In [19]:
# Reset for demo
TODO_STORE.clear()

# Test the coordinator with a complex task
result = coordinator_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Create a comprehensive morning routine guide for better energy.
        
The guide should:
1. Research the science behind morning routines
2. Include practical steps for exercise, nutrition, and mindset
3. Be saved as a well-formatted markdown file"""
    }]
})

print("Coordinator response:")
print(result["messages"][-1].content)

Coordinator response:
## ✅ Project Complete!

I've successfully created your comprehensive morning routine guide for better energy! Here's what was accomplished:

### 🔬 **Research Phase**
- Conducted in-depth research on circadian rhythm science, exercise physiology, nutrition timing, and psychology of morning habits
- Analyzed evidence-based benefits of different morning practices
- Identified key mechanisms for energy optimization

### 📝 **Guide Creation**
The complete guide includes:

**🏗️ Six Science-Based Pillars:**
1. **Sleep & Wake-up Optimization** - Circadian rhythm alignment
2. **Strategic Hydration** - Immediate rehydration protocols
3. **Strategic Light Exposure** - Natural energy activation
4. **Movement and Exercise** - Multiple intensity options
5. **Nutrition Timing and Quality** - Balanced breakfast guidelines
6. **Mindset and Mental Preparation** - Gratitude, intention-setting, mindfulness

**⏰ Three Complete Sample Timelines:**
- 30-minute Express Routine (for busy s

In [20]:
# Check the results
print("Final todo status:")
print(list_todos.invoke({}))

print("\nGenerated files in workspace:")
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

Final todo status:
✅ [todo_1] Research the science behind morning routines (completed)
✅ [todo_3] Create comprehensive morning routine guide (completed)
✅ [todo_5] Save the guide as a markdown file (completed)

Generated files in workspace:
  [FILE] morning_routine_energy_guide.md (16151 bytes)
  [FILE] morning_routine_guide.md (21260 bytes)
  [FILE] personalized_sleep_improvement_plan.md (5378 bytes)
  [DIR] research/
  [FILE] sleep_improvement_research.md (11993 bytes)


## Task 7: Long-term Memory Integration

The fourth key element is **Long-term Memory**. Deep Agents integrate with LangGraph's Store for persistent memory across sessions.

### Memory Types in Deep Agents

| Type | Scope | Use Case |
|------|-------|----------|
| **Thread Memory** | Single conversation | Current session context |
| **User Memory** | Across threads, per user | User preferences, history |
| **Shared Memory** | Across all users | Common knowledge, learned patterns |

### Integration with LangGraph Store

Deep Agents can use the same `InMemoryStore` (or `PostgresStore`) we learned in Session 6:

In [21]:
from langgraph.store.memory import InMemoryStore

# Create a memory store
memory_store = InMemoryStore()

# Store user profile
user_id = "user_alex"
profile_namespace = (user_id, "profile")

memory_store.put(profile_namespace, "name", {"value": "Alex"})
memory_store.put(profile_namespace, "goals", {
    "primary": "improve energy levels",
    "secondary": "better sleep"
})
memory_store.put(profile_namespace, "conditions", {
    "dietary": ["vegetarian"],
    "medical": ["mild anxiety"]
})
memory_store.put(profile_namespace, "preferences", {
    "exercise_time": "morning",
    "communication_style": "detailed"
})

print(f"Stored profile for {user_id}")

# Retrieve and display
for item in memory_store.search(profile_namespace):
    print(f"  {item.key}: {item.value}")

Stored profile for user_alex
  name: {'value': 'Alex'}
  goals: {'primary': 'improve energy levels', 'secondary': 'better sleep'}
  conditions: {'dietary': ['vegetarian'], 'medical': ['mild anxiety']}
  preferences: {'exercise_time': 'morning', 'communication_style': 'detailed'}


In [22]:
# Create memory-aware tools
from langgraph.store.base import BaseStore

@tool
def get_user_profile(user_id: str) -> str:
    """Retrieve a user's wellness profile from long-term memory.
    
    Args:
        user_id: The user's unique identifier
    
    Returns:
        User profile as formatted text
    """
    namespace = (user_id, "profile")
    items = list(memory_store.search(namespace))
    
    if not items:
        return f"No profile found for {user_id}"
    
    result = [f"Profile for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result)

@tool
def save_user_preference(user_id: str, key: str, value: str) -> str:
    """Save a user preference to long-term memory.
    
    Args:
        user_id: The user's unique identifier
        key: The preference key
        value: The preference value
    
    Returns:
        Confirmation message
    """
    namespace = (user_id, "preferences")
    memory_store.put(namespace, key, {"value": value})
    return f"Saved preference '{key}' for {user_id}"

print("Memory tools defined!")

Memory tools defined!


In [23]:
# Create a memory-enhanced agent
memory_tools = [
    get_user_profile,
    save_user_preference,
    write_todos,
    update_todo,
    list_todos,
]

memory_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=memory_tools,
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a Personal Wellness Assistant with long-term memory.

At the start of each conversation:
1. Check the user's profile to understand their goals and conditions
2. Personalize all advice based on their profile
3. Save any new preferences they mention

Always reference stored information to show you remember the user."""
)

print("Memory-enhanced agent created!")

Memory-enhanced agent created!


In [24]:
# Test the memory agent
TODO_STORE.clear()

result = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Hi! My user_id is user_alex. What exercise routine would you recommend for me?"
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Great to see you again, Alex! Based on your profile, I can see your primary goal is to improve energy levels with a secondary goal of better sleep. I also note you prefer morning exercise and have mild anxiety to consider.

Here's a personalized exercise routine that aligns perfectly with your goals:

## Morning Energy-Boosting Routine (30-40 minutes)

### **Week Structure:**
- **Monday, Wednesday, Friday:** Cardio + Light Strength
- **Tuesday, Thursday:** Yoga/Stretching + Core
- **Weekend:** Active recovery (walking, light activities)

### **Cardio + Strength Days (Mon/Wed/Fri):**
1. **Warm-up (5 min):** Light walking or marching in place
2. **Cardio (15-20 min):** 
   - Brisk walking, cycling, or dancing
   - Keep intensity moderate - you should be able to hold a conversation
3. **Strength (10-15 min):**
   - Bodyweight exercises: squats, modified push-ups, lunges
   - Use resistance bands for variety
   - Focus on major muscle groups

### **Yoga + Core Days (Tue/Thu

## Task 8: Skills - On-Demand Capabilities

**Skills** are a powerful feature for progressive capability disclosure. Instead of loading all tools upfront, agents can load specialized capabilities on demand.

### Why Skills?

1. **Context Efficiency**: Don't waste context on unused tool descriptions
2. **Specialization**: Skills can include detailed instructions for specific tasks
3. **Modularity**: Easy to add/remove capabilities
4. **Discoverability**: Agent can browse available skills

### SKILL.md Format

Skills are defined in markdown files with YAML frontmatter:

```markdown
---
name: skill-name
description: What this skill does
version: 1.0.0
tools:
  - tool1
  - tool2
---

# Skill Instructions

Detailed steps for how to use this skill...
```

In [25]:
# Let's look at the skills we created
skills_dir = Path("skills")

print("Available skills:")
for skill_dir in skills_dir.iterdir():
    if skill_dir.is_dir():
        skill_file = skill_dir / "SKILL.md"
        if skill_file.exists():
            content = skill_file.read_text()
            # Extract name and description from frontmatter
            lines = content.split("\n")
            name = ""
            desc = ""
            for line in lines:
                if line.startswith("name:"):
                    name = line.split(":", 1)[1].strip()
                if line.startswith("description:"):
                    desc = line.split(":", 1)[1].strip()
            print(f"  - {name}: {desc}")

Available skills:
  - meal-planning: Create personalized meal plans based on dietary needs and preferences
  - wellness-assessment: Assess user wellness goals and create personalized recommendations


In [26]:
# Read the wellness-assessment skill
skill_content = Path("skills/wellness-assessment/SKILL.md").read_text()
print(skill_content)

---
name: wellness-assessment
description: Assess user wellness goals and create personalized recommendations
version: 1.0.0
tools:
  - read_file
  - write_file
---

# Wellness Assessment Skill

You are conducting a comprehensive wellness assessment. Follow these steps:

## Step 1: Gather Information
Ask the user about:
- Current health goals (weight, fitness, stress, sleep)
- Any medical conditions or limitations
- Current exercise routine (or lack thereof)
- Dietary preferences and restrictions
- Sleep patterns and quality
- Stress levels and sources

## Step 2: Analyze Responses
Review the user's answers and identify:
- Primary wellness priority
- Secondary goals
- Potential barriers to success
- Existing healthy habits to build on

## Step 3: Create Assessment Report
Write a wellness assessment report to `workspace/wellness_assessment.md` containing:
- Summary of current wellness state
- Identified strengths
- Areas for improvement
- Recommended focus areas (prioritized)
- Suggeste

In [27]:
# Create a skill-aware tool
@tool
def load_skill(skill_name: str) -> str:
    """Load a skill's instructions for a specialized task.
    
    Available skills:
    - wellness-assessment: Assess user wellness and create recommendations
    - meal-planning: Create personalized meal plans
    
    Args:
        skill_name: Name of the skill to load
    
    Returns:
        Skill instructions
    """
    skill_path = Path(f"skills/{skill_name}/SKILL.md")
    if not skill_path.exists():
        available = [d.name for d in Path("skills").iterdir() if d.is_dir()]
        return f"Skill '{skill_name}' not found. Available: {', '.join(available)}"
    
    return skill_path.read_text()

print("Skill loader defined!")

Skill loader defined!


In [28]:
# Create an agent that can load and use skills
skill_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        load_skill,
        write_todos,
        update_todo,
        list_todos,
    ],
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a wellness assistant with access to specialized skills.

When a user asks for something that matches a skill:
1. Load the appropriate skill using load_skill()
2. Follow the skill's instructions carefully
3. Save outputs as specified in the skill

Available skills:
- wellness-assessment: For comprehensive wellness evaluations
- meal-planning: For creating personalized meal plans

If no skill matches, use your general wellness knowledge."""
)

print("Skill-aware agent created!")

Skill-aware agent created!


In [29]:
# Test with a skill-appropriate request
TODO_STORE.clear()

result = skill_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "I'd like a wellness assessment. I'm a 35-year-old office worker who sits most of the day, has trouble sleeping, and wants to lose 15 pounds. I'm vegetarian and have no major health conditions."
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## Your Personalized Wellness Assessment Summary

Based on your profile as a 35-year-old office worker with sleep challenges and a 15-pound weight loss goal, I've created a comprehensive assessment that identifies **sleep optimization** as your highest priority, as it will positively impact both your weight loss and energy levels.

### Key Findings:
- Your sedentary work lifestyle is likely contributing to both sleep issues and weight retention
- Your vegetarian diet provides a good foundation for healthy weight loss
- The absence of major health conditions gives you flexibility in your approach

### Your Action Plan:

**🔴 Start Today:**
1. Set a consistent bedtime routine with screen-free wind-down time
2. Take 2-3 minute movement breaks every hour during work  
3. Eat dinner 3+ hours before bedtime with adequate protein

**🟡 Next 1-2 Weeks:**
1. Establish 20-30 minute workouts 4x per week (mix cardio + strength)
2. Optimize your workspace ergonomics 
3. Track your sle

## Task 9: Using deepagents-cli

The `deepagents-cli` provides an interactive terminal interface for working with Deep Agents.

### Installation

```bash
uv pip install deepagents-cli
# or
pip install deepagents-cli
```

### Key Features

| Feature | Description |
|---------|-------------|
| **Interactive Sessions** | Chat with your agent in the terminal |
| **Conversation Resume** | Pick up where you left off |
| **Human-in-the-Loop** | Approve or reject agent actions |
| **File System Access** | Agent can read/write to your filesystem |
| **Remote Sandboxing** | Run in isolated Docker containers |

### Basic Usage

```bash
# Start an interactive session
deepagents

# Resume a previous conversation
deepagents --resume

# Use a specific model
deepagents --model openai:gpt-4o

# Enable human-in-the-loop approval
deepagents --approval-mode full
```

### Example Session

```
$ deepagents

Welcome to Deep Agents CLI!

You: Create a 7-day meal plan for a vegetarian athlete

Agent: I'll create a comprehensive meal plan for you. Let me:
1. Research vegetarian athlete nutrition needs
2. Design balanced daily menus
3. Save the plan to a file

[Agent uses tools...]

Agent: I've created your meal plan! You can find it at:
workspace/vegetarian_athlete_meal_plan.md

You: /exit
```

In [37]:
# Check if CLI is installed
import subprocess

try:
    result = subprocess.run(["deepagents", "--version"], capture_output=True, text=True)
    print(f"deepagents-cli version: {result.stdout.strip()}")
except FileNotFoundError:
    print("deepagents-cli not installed. Install with:")
    print("  uv pip install deepagents-cli")
    print("  # or")
    print("  pip install deepagents-cli")

deepagents-cli version: deepagents 0.0.17


### Try It Yourself!

After installing the CLI, try these commands in your terminal:

```bash
# Basic interactive session
deepagents

# With a specific working directory
deepagents --workdir ./workspace

# See all options
deepagents --help
```

Sample prompts to try:
1. "Create a weekly workout plan and save it to a file"
2. "Research the health benefits of meditation and summarize in a report"
3. "Analyze my current diet and suggest improvements" (then provide details)

## Task 10: Building a Complete Deep Agent System

Now let's bring together all four elements to build a comprehensive "Wellness Coach" system:

1. **Planning**: Track multi-week wellness programs
2. **Context Management**: Store session notes and progress
3. **Subagent Spawning**: Delegate to specialists (exercise, nutrition, mindfulness)
4. **Long-term Memory**: Remember user preferences and history

In [31]:
# Define specialized wellness subagents
# Subagents inherit the backend from the parent, so they use the same workspace
exercise_specialist = {
    "name": "exercise-specialist",
    "description": "Expert in exercise science, workout programming, and physical fitness. Use for exercise-related questions and plan creation.",
    "system_prompt": """You are an exercise specialist with expertise in:
- Workout programming for different fitness levels
- Exercise form and safety
- Progressive overload principles
- Recovery and injury prevention

Always consider the user's fitness level and any physical limitations.
Provide clear, actionable exercise instructions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

nutrition_specialist = {
    "name": "nutrition-specialist",
    "description": "Expert in nutrition science, meal planning, and dietary optimization. Use for food-related questions and meal plans.",
    "system_prompt": """You are a nutrition specialist with expertise in:
- Macro and micronutrient balance
- Meal planning and preparation
- Dietary restrictions and alternatives
- Nutrition timing for performance

Always respect dietary restrictions and preferences.
Focus on practical, achievable meal suggestions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

mindfulness_specialist = {
    "name": "mindfulness-specialist",
    "description": "Expert in stress management, sleep optimization, and mental wellness. Use for stress, sleep, and mental health questions.",
    "system_prompt": """You are a mindfulness and mental wellness specialist with expertise in:
- Stress reduction techniques
- Sleep hygiene and optimization
- Meditation and breathing exercises
- Work-life balance strategies

Be supportive and non-judgmental.
Provide practical techniques that can be implemented immediately.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Specialist subagents defined!")

Specialist subagents defined!


In [32]:
# Create the Wellness Coach coordinator
wellness_coach = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        # Planning
        write_todos,
        update_todo,
        list_todos,
        # Long-term Memory
        get_user_profile,
        save_user_preference,
        # Skills
        load_skill,
    ],
    backend=filesystem_backend,  # All file ops go to workspace
    subagents=[exercise_specialist, nutrition_specialist, mindfulness_specialist],
    system_prompt="""You are a Personal Wellness Coach that coordinates comprehensive wellness programs.

## Your Role
- Understand each user's unique goals, constraints, and preferences
- Create personalized, multi-week wellness programs
- Coordinate between exercise, nutrition, and mindfulness specialists
- Track progress and adapt recommendations

## Workflow
1. **Initial Assessment**: Get user profile and understand their situation
2. **Planning**: Create a todo list for the program components
3. **Delegation**: Use specialists for domain-specific content:
   - exercise-specialist: Workout plans and fitness guidance
   - nutrition-specialist: Meal plans and dietary advice
   - mindfulness-specialist: Stress and sleep optimization
4. **Integration**: Combine specialist outputs into a cohesive program
5. **Documentation**: Save all plans and recommendations to files

## Important
- Always check user profile first for context
- Respect any medical conditions or dietary restrictions
- Provide clear, actionable recommendations
- Save progress to files so users can reference later"""
)

print("Wellness Coach created with all 4 Deep Agent elements!")

Wellness Coach created with all 4 Deep Agent elements!


In [33]:
# Test the complete system
TODO_STORE.clear()

result = wellness_coach.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. I'd like you to create a 2-week wellness program for me.

I want to focus on:
1. Building a consistent exercise routine (I can exercise 3x per week for 30 mins)
2. Improving my diet (remember I'm vegetarian)
3. Better managing my work stress and improving my sleep

Please create comprehensive plans for each area and save them as separate files I can reference."""
    }]
})

print("Wellness Coach response:")
print(result["messages"][-1].content)

Wellness Coach response:
Perfect! Your comprehensive 2-week wellness program is complete and ready to go. Here's what I've created for you:

## 📁 Your Complete Wellness Program Files

I've saved **4 separate reference files** you can access anytime:

1. **`/alex_exercise_program.md`** - Your 3x/week, 30-minute morning workout routine
2. **`/alex_nutrition_plan.md`** - Complete vegetarian meal plans with recipes and shopping lists  
3. **`/alex_stress_sleep_program.md`** - Daily stress management and sleep optimization techniques
4. **`/alex_integrated_wellness_schedule.md`** - Master day-by-day schedule combining all three programs

## 🎯 Program Highlights

**Tailored to Your Profile:**
- ✅ Vegetarian nutrition plan with anxiety-reducing foods
- ✅ Morning workout preference accommodated
- ✅ Gentle, beginner-friendly exercises suitable for mild anxiety
- ✅ Energy-boosting foods and sleep-promoting nutrients
- ✅ Practical stress management for working professionals

**Week 1 Focus:** Bui

In [34]:
# Review what was created
print("=" * 60)
print("FINAL TODO STATUS")
print("=" * 60)
print(list_todos.invoke({}))

print("\n" + "=" * 60)
print("GENERATED FILES")
print("=" * 60)
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

FINAL TODO STATUS
✅ [todo_1] Create Exercise Program (completed)
✅ [todo_3] Develop Nutrition Plan (completed)
✅ [todo_5] Design Stress Management & Sleep Program (completed)
✅ [todo_7] Integrate All Programs (completed)
✅ [todo_9] Save Programs to Files (completed)

GENERATED FILES
  [FILE] alex_exercise_program.md (3948 bytes)
  [FILE] alex_integrated_wellness_schedule.md (9735 bytes)
  [FILE] alex_nutrition_plan.md (6109 bytes)
  [FILE] alex_stress_sleep_program.md (7302 bytes)
  [DIR] exercise_programs/
  [FILE] meal_prep_tips.txt (966 bytes)
  [FILE] morning_routine_energy_guide.md (16151 bytes)
  [FILE] morning_routine_guide.md (21260 bytes)
  [FILE] personalized_sleep_improvement_plan.md (5378 bytes)
  [DIR] research/
  [FILE] shopping_list_week1.txt (629 bytes)
  [FILE] shopping_list_week2.txt (455 bytes)
  [FILE] sleep_improvement_research.md (11993 bytes)
  [FILE] stress_management_sleep_improvement_program.txt (4738 bytes)
  [FILE] vegetarian_meal_plan_week1.txt (2000 bytes)

In [35]:
# Read one of the generated files
files = list(WORKSPACE.glob("*.md"))
if files:
    print(f"\nContents of {files[0].name}:")
    print("=" * 60)
    print(files[0].read_text()[:2000] + "..." if len(files[0].read_text()) > 2000 else files[0].read_text())


Contents of sleep_improvement_research.md:
# Comprehensive Sleep Improvement Research: Evidence-Based Strategies

## Executive Summary

This research document addresses three specific sleep challenges with scientifically-backed solutions:
1. **Inconsistent bedtime schedule (10pm-1am variation)**
2. **Phone use in bed**
3. **Morning fatigue**

The strategies are organized into immediate interventions and gradual improvements, with explanations of underlying mechanisms.

---

## Problem Analysis & Scientific Context

### 1. Inconsistent Bedtime Schedule Impact
**Research Finding**: Variable sleep timing disrupts the circadian clock, leading to:
- Reduced sleep efficiency (study: Phillips et al., 2017)
- Increased cortisol variability (Zeitzer et al., 2018)
- Impaired glucose metabolism (Scheer et al., 2009)
- Social jet lag effects (Roenneberg et al., 2012)

**Mechanism**: The suprachiasmatic nucleus (SCN) requires consistent timing cues to maintain proper circadian rhythm alignment.

#

---
## Question #3:

What are the key considerations when designing **subagent configurations**?

Consider:
- When should subagents share tools vs have distinct tools?
- How do you decide which model to use for each subagent?
- What's the right granularity for subagent specialization?

##### Answer:
Subagent design is about containment, not intelligence.

> Tool sharing vs isolation
- Share tools when tasks require common infrastructure (search, vector DB)
- Isolate tools when misuse risk exists (code execution, shell access)

> Model selection
- Lightweight models for extractive or repetitive tasks
- Stronger models for planning, synthesis, or verification
- Supervisor agents should usually be the most reliable model

> Specialization granularity
- Subagents should own one responsibility
- If a subagent needs internal planning, it’s probably too broad
- If it only reformats text, it’s probably too narrow

Good subagents feel like microservices, not helpers.

## Question #4:

For a **production wellness application** using Deep Agents, what would you need to add?

Consider:
- Safety guardrails for health advice
- Persistent storage (not in-memory)
- Multi-user support and isolation
- Monitoring and observability
- Cost management with subagents

##### Answer:
A production system needs safeguards beyond agent logic.

> Safety guardrails
- Hard boundaries on medical advice
- Refusal and escalation paths
- Clear “not a doctor” policy enforcement

> Persistent storage
- Replace in-memory stores with durable databases
- Version memory schemas
- Encrypt sensitive user data

> Multi-user isolation
- Per-user namespaces
- Thread-scoped execution
- No shared memory leakage

> Monitoring & observability
- Trace agent decisions and tool calls
- Log todo completion rates
- Detect looping or hallucinations

> Cost controls
- Cap subagent spawn depth
- Route simple tasks to cheaper models
- Cache repeated retrievals

---
## Activity #2: Build a Wellness Coach Agent

Build your own wellness coach that uses all 4 Deep Agent elements.

### Requirements:
1. **Planning**: Create todos for a 30-day wellness challenge
2. **Context Management**: Store daily check-in notes
3. **Subagents**: At least 2 specialized subagents
4. **Memory**: Remember user preferences across interactions

### Challenge:
Create a "30-Day Wellness Challenge" system that:
- Generates a personalized 30-day plan
- Tracks daily progress
- Adapts recommendations based on feedback
- Saves a weekly summary report

In [80]:
### YOUR CODE HERE ###

from pathlib import Path
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent

from langchain_core.tools import tool

@tool
def safe_write_file(path: str, content: str) -> str:
    """
    Write a file ONLY to a relative workspace path.
    Reject absolute paths like /something.md (root filesystem is read-only).
    """
    if path.startswith("/"):
        return "ERROR: Absolute paths are not allowed. Use a relative path like users/<id>/plans/...."
    return write_file.invoke({"path": path, "content": content})




# -----------------------------
# Extra tools for the challenge system
# -----------------------------
@tool
def log_daily_checkin(user_id: str, day: int, note: str) -> str:
    """Log daily check-in notes for the user
    """
    day = int(day)
    path = f"users/{user_id}/checkins/day_{day:02d}.md"
    content = f"# Day {day} Check-in\n\n{note}\n"
    return safe_write_file.invoke({"path": path, "content": content})


@tool
def get_recent_checkins(user_id: str, last_n_days: int = 3) -> str:
    """
    Retrieve the most recent N check-ins from the workspace to help adapt the plan.
    """
    last_n_days = int(last_n_days)
    base = Path("workspace") / "users" / user_id / "checkins"
    if not base.exists():
        return f"No checkins found yet for {user_id}."

    files = sorted(base.glob("day_*.md"))
    if not files:
        return f"No checkins found yet for {user_id}."

    recent = files[-last_n_days:]
    parts = []
    for f in recent:
        parts.append(f"\n---\n## {f.name}\n{f.read_text()}\n")
    return "".join(parts).strip()



In [81]:
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent

nutrition_subagent = {
    "name": "nutrition-coach",
    "description": "Specialist for meal planning, macros, hydration, and realistic food habits.",
    "system_prompt": """
You are a Nutrition Coach subagent.
Give practical vegetarian-friendly meal templates, hydration tips, and easy dinner ideas.
Avoid medical claims. Output concise, actionable steps.
"""
}

sleep_stress_subagent = {
    "name": "sleep-stress-coach",
    "description": "Specialist for sleep hygiene, stress management, routines, and recovery habits.",
    "system_prompt": """
You are a Sleep & Stress Coach subagent.
Provide evidence-based sleep hygiene and stress reduction routines.
Avoid medical diagnosis. Be practical and structured.
"""
}

# ✅ All outputs are forced into these workspace-relative paths
PLAN_PATH = "users/{user_id}/plans/30_day_challenge.md"
CHECKLIST_PATH = "users/{user_id}/plans/daily_checklist.md"
ADAPT_PATH = "users/{user_id}/plans/adaptations.md"

wellness_coach_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        # Planning
        write_todos, update_todo, list_todos,

        # Long-term memory tools (already in notebook)
        get_user_profile, save_user_preference,

        # Context management
        log_daily_checkin, get_recent_checkins,

        # Workspace tools
        ls, read_file, edit_file,

        # ✅ Safe writer only
        safe_write_file,
    ],
    backend=filesystem_backend,
    subagents=[nutrition_subagent, sleep_stress_subagent],
    system_prompt=f"""
You are a 30-Day Wellness Challenge Coach.

NON-NEGOTIABLE FILE RULE:
- You may ONLY save files using safe_write_file.
- NEVER write to absolute paths like /something.md (root FS is read-only).
- All saved files must be under users/<user_id>/...

You MUST use all 4 Deep Agent elements:
1) Planning: Create todos for the full 30-day challenge (write_todos) and update progress.
2) Context Management: Store each daily check-in (log_daily_checkin) and use them to adapt.
3) Subagents: Delegate nutrition to nutrition-coach and sleep/stress to sleep-stress-coach.
4) Memory: Persist stable user preferences using save_user_preference and retrieve using get_user_profile.

When user asks to start a challenge:
- Extract user_id from the message.
- Ask for missing constraints if needed (time availability, dietary preference, routine preference).
- Save stable preferences (vegetarian, morning routine, quick dinners, goals) via save_user_preference.
- Create a personalized 30-day plan with weekly themes.
- Save plan to: users/<user_id>/plans/30_day_challenge.md
- Save checklist to: users/<user_id>/plans/daily_checklist.md

When user submits a check-in:
- Log it with log_daily_checkin(user_id, day, note)
- Load recent check-ins with get_recent_checkins
- Adapt next 3 days only
- Save adaptations to: users/<user_id>/plans/adaptations.md

Safety:
- No medical diagnosis. If red flags appear, suggest a professional.
"""
)

In [ ]:
TODO_STORE.clear()

start_msg = """
Hi! My user_id is user_demo.
I want a 30-day wellness challenge.
Constraints:
- I can do 25 minutes per day on weekdays, 45 mins on weekends
- I’m vegetarian
- I prefer morning routines
- I’m mainly focused on better sleep and reduced stress
Please generate my plan and store it.
Also remember my preferences for next time.
"""

result = wellness_coach_agent.invoke({"messages": [{"role": "user", "content": start_msg}]})
print(result["messages"][-1].content)

print("\nFILES:\n", ls.invoke({"path": "users/user_demo"}))
print("\nPLAN PREVIEW:\n", read_file.invoke({"path": "users/user_demo/plans/30_day_challenge.md"}))
print("\nCHECKLIST PREVIEW:\n", read_file.invoke({"path": "users/user_demo/plans/daily_checklist.md"}))


---
## Summary

In this session, we explored **Deep Agents** and their four key elements:

| Element | Purpose | Implementation |
|---------|---------|----------------|
| **Planning** | Track complex tasks | `write_todos`, `update_todo`, `list_todos` |
| **Context Management** | Handle large contexts | File system tools, automatic offloading |
| **Subagent Spawning** | Delegate to specialists | `task` tool with custom configs |
| **Long-term Memory** | Remember across sessions | LangGraph Store integration |

### Key Takeaways:

1. **Deep Agents handle complexity** - They can manage long-horizon, multi-step tasks effectively
2. **Planning is context engineering** - Todo lists and files serve as extended memory for the agent
3. **Subagents prevent context bloat** - Delegation keeps the main agent focused and efficient
4. **Skills enable progressive disclosure** - Load capabilities on-demand instead of upfront
5. **The CLI makes interaction natural** - Interactive sessions with conversation resume

### Further Reading

- [Deep Agents Documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents GitHub](https://github.com/langchain-ai/deepagents)
- [Context Management Blog Post](https://www.blog.langchain.com/context-management-for-deepagents/)
- [Building Multi-Agent Applications](https://www.blog.langchain.com/building-multi-agent-applications-with-deep-agents/)
- [LangGraph Memory Concepts](https://langchain-ai.github.io/langgraph/concepts/memory/)